# Vector Databases & ANN Indexes: search millions of vectors in milliseconds

Retrieval is nearest-neighbour search in embedding space (chapter 3). The catch: exact ("flat") search compares the query to **every** vector — O(N·d) per query — which is fine for thousands but brutal for millions (10M × 768-dim ≈ **7.7 billion** multiply-adds *per query*).

**Approximate Nearest Neighbour (ANN)** indexes fix this by skipping almost all vectors — routing the query to the right neighbourhood — trading a tunable bit of **recall** for orders-of-magnitude speed. This notebook builds an **IVF** index from scratch (k-means cells you probe) and *measures* the recall/speed tradeoff, exposing the **recall cliff**. We import the canonical functions from [`vector_indexes.py`](vector_indexes.py); the demo is **seeded and deterministic**, so every number is identical on every run.

**By the end you'll have:** felt the brute-force cost, built an IVF index, swept `nprobe` to see the recall cliff and the speedup, watched greedy graph search walk downhill (the HNSW mechanic), and computed PQ's memory compression.

## Step 1 — set up and feel the brute-force cost

Build a toy corpus of 20,000 lightly-clustered 64-dim vectors plus 200 queries. Then count the brute-force cost: every query scans every vector. On the toy it's ~1.3M multiply-adds; at a real 10M×768 corpus it's ~7.7 **billion** per query — the wall ANN exists to clear.

In [1]:
import numpy as np

from vector_indexes import (
    make_dataset, brute_force_topk, build_ivf, ivf_search,
    evaluate_ivf_sweep, recall_at_k, build_knn_graph, greedy_graph_search,
    pq_compression_ratio, NLIST, TOP_K, NPROBE_SWEEP,
)
try:
    import torch
    print('torch:', torch.__version__)
except ImportError:
    pass
print('numpy:', np.__version__)

corpus, queries = make_dataset()
print(f'corpus: {corpus.shape[0]:,} vectors x {corpus.shape[1]} dims | queries: {len(queries)}')
toy_ops = corpus.shape[0] * corpus.shape[1]
print(f'brute force per query: {corpus.shape[0]:,} x {corpus.shape[1]} = {toy_ops:,} multiply-adds')
print(f'at 10M x 768-dim that would be {10_000_000 * 768:,} (~7.7B) per query')

torch: 2.12.0
numpy: 2.4.6
corpus: 20,000 vectors x 64 dims | queries: 200
brute force per query: 20,000 x 64 = 1,280,000 multiply-adds
at 10M x 768-dim that would be 7,680,000,000 (~7.7B) per query


## Step 2 — exact ground truth (the baseline to measure recall against)

To know whether an *approximate* index is good, we need the *exact* answer to compare against. `brute_force_topk` computes the true top-10 for each query by scanning all vectors — the O(N·d) baseline. This is also what an index must beat on speed while staying close on recall.

In [2]:
ground_truth = np.array([brute_force_topk(q, corpus) for q in queries])
print('ground-truth shape (queries, k):', ground_truth.shape)
print('exact top-10 for query 0:', ground_truth[0])

ground-truth shape (queries, k): (200, 10)
exact top-10 for query 0: [ 5268 16797  9146  4903 17339 19031 12014 11204  4511  6398]


## Step 3 — build the IVF index (k-means cells + inverted lists)

IVF partitions the corpus into `nlist` Voronoi cells with k-means, then stores an inverted list mapping each cell to its vectors — done once, offline. At query time we'll probe only a few cells instead of scanning everything. Build it and check the shapes.

In [3]:
index = build_ivf(corpus)   # k-means into nlist cells
print(f'centroids shape (nlist, dim): {index.centroids.shape}')
cell_sizes = [len(ids) for ids in index.cells.values()]
print(f'nlist={NLIST} cells | mean cell size: {np.mean(cell_sizes):.0f} vectors '
      f'(min {min(cell_sizes)}, max {max(cell_sizes)})')
assert index.centroids.shape[0] == NLIST, 'should have NLIST centroids'

centroids shape (nlist, dim): (64, 64)
nlist=64 cells | mean cell size: 312 vectors (min 80, max 661)


## Step 4 — one query: probe a few cells, scan only those

`ivf_search` finds the `nprobe` nearest cell centroids, then exact-searches only the vectors in those cells. With `nprobe=8` it scans far fewer than all 20,000 vectors — that's the speedup. Check how many it actually scanned, and that the assert confirms it's a small fraction.

In [4]:
retrieved, n_scanned = ivf_search(index, queries[0], k=TOP_K, nprobe=8)
print(f'nprobe=8 scanned {n_scanned:,} of {len(corpus):,} vectors '
      f'({100 * n_scanned / len(corpus):.0f}% of the corpus)')
print(f'recall@10 for this query: {recall_at_k(retrieved, ground_truth[0]):.2f}')
assert n_scanned < len(corpus) // 4, 'nprobe=8 should scan well under a quarter of the corpus'

nprobe=8 scanned 1,413 of 20,000 vectors (7% of the corpus)
recall@10 for this query: 1.00


## Step 5 — sweep nprobe: the recall cliff

Now the headline. Sweep `nprobe` from 1 to 64 (= nlist) and measure mean recall@10 and the % of the corpus scanned. **Read the table:** at `nprobe=1` you scan ~2% (≈57× faster) but recall is only ~0.37 — you miss most neighbours. Recall recovers fast as you probe more, reaching ~1.0 by `nprobe=16`. The **sweet spot is `nprobe=8`**: recall ~0.99 at ~13% scanned (~8× faster).

In [5]:
sweep = evaluate_ivf_sweep(index, queries, ground_truth)
print(f"{'nprobe':>6} | {'recall@10':>9} | {'% scanned':>9} | {'~speedup':>8}")
print('-' * 42)
for nprobe, (recall, frac) in sweep.items():
    print(f'{nprobe:>6} | {recall:>9.3f} | {frac * 100:>8.1f}% | {1 / frac:>7.1f}x')

# correctness BEFORE any claim: the cliff must be real and recall must be monotone in nprobe
recalls = [sweep[n][0] for n in NPROBE_SWEEP]
assert recalls[0] < 0.6, 'nprobe=1 should miss many neighbours (the cliff)'
assert recalls[-1] >= 0.999, 'probing all cells recovers exact recall'
assert all(a <= b + 1e-9 for a, b in zip(recalls, recalls[1:])), 'recall must rise with nprobe'
sweet = next(n for n in NPROBE_SWEEP if sweep[n][0] >= 0.95)
print(f'\nrecall climbs {recalls[0]:.3f} -> {recalls[-1]:.3f}; >=0.95 first at nprobe={sweet} '
      f'(~{1 / sweep[sweet][1]:.0f}x faster than flat)')

nprobe | recall@10 | % scanned | ~speedup
------------------------------------------
     1 |     0.365 |      1.8% |    56.8x
     2 |     0.614 |      3.5% |    28.4x
     4 |     0.877 |      6.8% |    14.8x
     8 |     0.992 |     13.0% |     7.7x
    16 |     1.000 |     25.3% |     4.0x
    32 |     1.000 |     51.1% |     2.0x
    64 |     1.000 |    100.0% |     1.0x

recall climbs 0.365 -> 1.000; >=0.95 first at nprobe=8 (~8x faster than flat)


## Step 6 — the graph mechanic (simplified HNSW)

HNSW uses a navigable graph instead of cells: greedily hop neighbour→neighbour toward the query. A full HNSW is heavy, so we build a **simplified single-layer** k-NN graph and run greedy descent to show the core hop — each step strictly closer to the query, until no neighbour is closer (a local optimum). We trace one query (the figure's `query[3]`), then **aggregate over 50 queries** for the honest summary: the hops are few, but a single-layer graph lands far from the true neighbour (the local-optimum stall) — which is exactly *why* real HNSW adds a hierarchy.

In [6]:
graph_n = 2000
graph = build_knn_graph(corpus[:graph_n])   # each node linked to its 16 nearest (O(N^2) build, toy)
sub = corpus[:graph_n]

# one query, traced: the distance falls each hop until a local optimum (this is the figure's query[3])
node, hops = greedy_graph_search(sub, graph, queries[3], entry=0)
rank = int(np.where(np.argsort(((sub - queries[3]) ** 2).sum(1)) == node)[0][0])
print(f'single query (query[3]): {hops} hops downhill, landed at rank {rank} of {len(sub):,}')

# aggregate over 50 queries from one fixed entry point — the honest summary the page quotes
hops_list, landed_ranks = [], []
for qi in range(50):
    n, h = greedy_graph_search(sub, graph, queries[qi], entry=0)
    hops_list.append(h)
    full_order = np.argsort(((sub - queries[qi]) ** 2).sum(1))
    landed_ranks.append(int(np.where(full_order == n)[0][0]))  # rank of the landed node (0 = exact NN)
mean_hops = float(np.mean(hops_list))
median_rank = float(np.median(landed_ranks))
print(f'\nover 50 queries: mean {mean_hops:.1f} hops, median rank {median_rank:.0f} of {len(sub):,} '
      f'(top {100 * (median_rank + 1) / len(sub):.1f}%)')
print('the hop mechanic works; single-layer stalls in local optima -> HNSW adds hierarchy')
assert mean_hops >= 1.0, 'greedy search should take at least one downhill hop'

single query (query[3]): 2 hops downhill, landed at rank 323 of 2,000

over 50 queries: mean 1.9 hops, median rank 326 of 2,000 (top 16.4%)
the hop mechanic works; single-layer stalls in local optima -> HNSW adds hierarchy


## Step 7 — product quantization: the memory math

At billion scale, storing raw float32 vectors is too much RAM. **Product Quantization** splits each vector into `m` sub-vectors and replaces each with a small code id — shrinking a vector from `dim×32` bits to `m×nbits` bits. Compute the compression ratio for our toy.

In [7]:
raw_bits, pq_bits, ratio = pq_compression_ratio()
print(f'raw float32 vector: {raw_bits:,} bits ({raw_bits // 8} bytes)')
print(f'PQ code:            {pq_bits} bits ({pq_bits // 8} bytes)')
print(f'compression ratio:  {ratio:.0f}x smaller')
assert ratio > 1, 'PQ must compress'

raw float32 vector: 2,048 bits (256 bytes)
PQ code:            64 bits (8 bytes)
compression ratio:  32x smaller


## Try it yourself

Before you change anything, **predict**: the recall cliff above came from `nlist=64` cells. If you rebuild with **more** cells (say `nlist=256`) so each cell is *smaller*, what happens to recall at a fixed `nprobe=8` — does it go up or down?

Then try it: `build_ivf(corpus, n_cells=256)` and re-measure recall@10 at `nprobe=8`. *Hint:* more cells means each cell holds fewer vectors, so probing 8 of them now covers a *smaller* slice of the corpus — recall at fixed `nprobe` **drops** (you'd need a larger `nprobe` to compensate). That coupling — `nprobe` should scale with `nlist` — is the core IVF tuning insight: the two knobs trade off together, not independently.

> To see the real thing at scale, build a `faiss.IndexIVFFlat(quantizer, d, nlist)` and set `index.nprobe`, or a `faiss.IndexHNSWFlat(d, M)` and tune `efSearch` — same knobs, production speed.

## What we saw

- **Brute force is O(N·d)** — fine on the toy (~1.3M ops/query), hopeless at 10M×768 (~7.7B ops/query). That linear wall is why ANN exists.
- **IVF skips most vectors** — k-means cells let a query probe only `nprobe` nearby cells instead of scanning all N.
- **The recall cliff is the core tradeoff** — recall ~0.37 at nprobe=1 (≈57× faster, but misses neighbours) climbing to ~1.0; sweet spot nprobe=8 (~0.99 recall, ~8× faster).
- **Graph search walks downhill** — the greedy hop is the HNSW mechanic; a single-layer graph stalls in local optima, which is exactly why HNSW adds a hierarchy.
- **PQ trades memory for a little recall** — 32× smaller vectors via codes, for billion-scale.

**What we skipped** (and where to read it): the full HNSW $O(\log N)$ navigation; the **filtering + ANN** production trap; index build/memory blowups; and choosing a vector DB. All of that is in the [Vector Databases concept page](../04-Vector-Databases-and-ANN-Indexes.md), with its [curated references](../04-Vector-Databases-and-ANN-Indexes.references.md). Next: [hybrid search](../../05-Hybrid-Search-BM25-and-Dense/05-Hybrid-Search-BM25-and-Dense.md) combines this dense ANN search with lexical BM25 for the best of both.